# <center> Homework 110

In [1]:
from importlib import reload
import finetuning
import tensorflow as tf
import keras_tuner as kt
import numpy as np
import pandas as pd
import time
from copy import deepcopy

2026-01-19 15:33:02.030181: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-19 15:33:03.220454: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-19 15:33:07.801750: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## Task 0

In [4]:
t = tf.constant([[1., 2., 3.], [4., 5., 6.]])
tf.reduce_sum(t, axis=1)

<tf.Tensor: shape=(2,), dtype=float32, numpy=array([ 6., 15.], dtype=float32)>

In [67]:
print(t.shape)
t.dtype

(2, 3)


tf.float32

In [68]:
t[:, 1:]

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[2., 3.],
       [5., 6.]], dtype=float32)>

In [69]:
t[..., 1, tf.newaxis]

<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[2.],
       [5.]], dtype=float32)>

In [70]:
t + 10

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[11., 12., 13.],
       [14., 15., 16.]], dtype=float32)>

In [71]:
tf.square(t)

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[ 1.,  4.,  9.],
       [16., 25., 36.]], dtype=float32)>

In [72]:
t @ tf.transpose(t)

<tf.Tensor: shape=(2, 2), dtype=float32, numpy=
array([[14., 32.],
       [32., 77.]], dtype=float32)>

In [73]:
tf.constant(42)

<tf.Tensor: shape=(), dtype=int32, numpy=42>

In [74]:
a = np.array([2., 4., 5.])
tf.constant(a)

<tf.Tensor: shape=(3,), dtype=float64, numpy=array([2., 4., 5.])>

In [75]:
t.numpy()

array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)

In [76]:
np.square(t)

array([[ 1.,  4.,  9.],
       [16., 25., 36.]], dtype=float32)

In [ ]:
tf.constant(2.) + tf.constant(40)
# InvalidArgumentError: cannot compute AddV2 as input #1(zero-based) was expected to 
# be a float tensor but is a int32 tensor [Op:AddV2] name:

In [ ]:
tf.constant(2.) + tf.constant(40., dtype=tf.float64)
# InvalidArgumentError: cannot compute AddV2 as input #1(zero-based) 
# was expected to be a float tensor but is a double tensor [Op:AddV2] name:

In [79]:
t2 = tf.constant(40., dtype=tf.float64)
tf.constant(2.0) + tf.cast(t2, tf.float32)

<tf.Tensor: shape=(), dtype=float32, numpy=42.0>

In [80]:
v = tf.Variable([[1., 2., 3.], [4., 5., 6.]])
v

<tf.Variable 'Variable:0' shape=(2, 3) dtype=float32, numpy=
array([[1., 2., 3.],
       [4., 5., 6.]], dtype=float32)>

In [81]:
v.assign(2 * v)
# v now equals [[2., 4., 6.], [8., 10., 12.]]
v[0, 1].assign(42)
# v now equals [[2., 42., 6.], [8., 10., 12.]]
v[:, 2].assign([0., 1.]) # v now equals [[2., 42., 0.], [8., 10., 1.]]
v.scatter_nd_update(
# v now equals [[100., 42., 0.], [8., 10., 200.]]
indices=[[0, 0], [1, 2]], updates=[100., 200.])

<tf.Variable 'UnreadVariable' shape=(2, 3) dtype=float32, numpy=
array([[100.,  42.,   0.],
       [  8.,  10., 200.]], dtype=float32)>

## Task 1

In [2]:
np.random.seed(13)
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(return_X_y=True)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, random_state=42, test_size=0.2, shuffle=True) 
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, random_state=42, test_size=0.1, shuffle=True) 

In [5]:
def build_model(hp):
    n_neurons = hp.Int('n_neurons', 3, 5)
    n_hidden = hp.Choice('n_hidden', [3, 4])
    lr = hp.Float('lr', 0.001, 0.1, step=0.05)

    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(X_train.shape[1:]))
    norm = tf.keras.layers.Normalization()
    norm.adapt(X_train)
    model.add(norm)

    for _ in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons,
                                        activation='relu',
                                        kernel_initializer='he_normal'))

    model.add(tf.keras.layers.Dense(1))
    optimizer = tf.keras.optimizers.Adam(lr)
    model.compile(loss='mse',
                  optimizer=optimizer,
                  metrics=['RootMeanSquaredError'])
    
    return model

In [43]:
reload(finetuning)
from finetuning import GridSearch

tuner = GridSearch(build_model, 'val_RootMeanSquaredError', max_trials=3,
                   directory='gridsearach', project_name='test5')

tuner.search(X_train, y_train, epochs=5, validation_data=(X_val, y_val))


Trail 1/3
n_neurons                      3
n_hidden                       3
lr                             0.001

Epoch 1/5
461/465 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - RootMeanSquaredError: 1.9397 - loss: 3.8002
--------------------
Layer 1: Dense (dense_144) - W_mean: 0.11 b_mean: 0.10 - W_std: 0.50 b_std: 0.25
Layer 2: Dense (dense_145) - W_mean: -0.23 b_mean: 0.14 - W_std: 0.74 b_std: 0.13
Layer 3: Dense (dense_146) - W_mean: -0.17 b_mean: 0.09 - W_std: 0.96 b_std: 0.25
Layer 4: Dense (dense_147) - W_mean: 0.53 b_mean: 0.23 - W_std: 0.71 b_std: 0.00
--------------------

465/465 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - RootMeanSquaredError: 1.6304 - loss: 2.6582 - val_RootMeanSquaredError: 1.1596 - val_loss: 1.3448
Epoch 2/5
452/465 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - RootMeanSquaredError: 1.2388 - loss: 1.5458
--------------------
Layer 1: Dense (dense_144) - W_mean: 0.12 b_mean: 0.15 - W_std: 0.47 b_std: 0.25
Layer 2: Dense (dense_145) - W_mean: -0.25 b_mean: 0.17 - W_std: 0.74 b_std: 0.

In [44]:
reload(finetuning)
from finetuning import GridSearch

tuner = GridSearch(build_model, 'val_RootMeanSquaredError',
                   directory='gridsearach', project_name='test5')

tuner.search(X_train, y_train, epochs=5, validation_data=(X_val, y_val))


Trail 4/12
n_neurons                      3
n_hidden                       4
lr                             0.001

Epoch 1/5
451/465 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - RootMeanSquaredError: 1.8916 - loss: 3.6362
--------------------
Layer 1: Dense (dense_161) - W_mean: 0.04 b_mean: 0.02 - W_std: 0.53 b_std: 0.26
Layer 2: Dense (dense_162) - W_mean: 0.21 b_mean: 0.37 - W_std: 0.69 b_std: 0.02
Layer 3: Dense (dense_163) - W_mean: 0.18 b_mean: 0.19 - W_std: 0.77 b_std: 0.13
Layer 4: Dense (dense_164) - W_mean: 0.09 b_mean: 0.01 - W_std: 0.58 b_std: 0.20
Layer 5: Dense (dense_165) - W_mean: -0.28 b_mean: 0.24 - W_std: 0.63 b_std: 0.00
--------------------

465/465 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - RootMeanSquaredError: 1.5272 - loss: 2.3324 - val_RootMeanSquaredError: 0.9867 - val_loss: 0.9735
Epoch 2/5
451/465 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - RootMeanSquaredError: 0.9117 - loss: 0.8317
--------------------
Layer 1: Dense (dense_161) - W_mean: 0.02 b_mean: 0.06 - W_std: 0.54 b_std: 0.3

## Taks 2

In [5]:
fashion_mnist = tf.keras.datasets.fashion_mnist.load_data()
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist
X_train, y_train = X_train_full[:-5000], y_train_full[:-5000]
X_valid, y_valid = X_train_full[-5000:], y_train_full[-5000:]

X_train, X_valid, X_test = X_train / 255., X_valid / 255., X_test / 255.

In [61]:
def bn_convert(model):
    new_layers = []

    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization) and \
                isinstance(new_layers[-1], tf.keras.layers.Dense):
           
            W, b = new_layers[-1].get_weights()
            gamma, beta, mean, var = layer.get_weights()
            eps = layer.epsilon

            W_new = W * (gamma / np.sqrt(var + eps))
            b_new = (b - mean) * (gamma / np.sqrt(var + eps)) + beta
            
            new_layers[-1].set_weights([W_new, b_new])
            continue

        new_layers.append(layer)

    return tf.keras.models.Sequential(new_layers)

In [46]:
def build_model(hp):
    n_hidden = hp.Int("n_hidden", min_value=3, max_value=8)
    n_neurons = hp.Int("n_neurons", min_value=16, max_value=256, default=50)
    learning_rate = hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="log")

    optimizer_choice = hp.Choice('optimizer', ['sgd', 'adam', 'nadam', 'adamax'])
    match optimizer_choice:
      case 'nadam': optimizer = tf.keras.optimizers.Nadam(learning_rate)
      case 'adam': optimizer = tf.keras.optimizers.Adam(learning_rate)
      case 'adamax': optimizer = tf.keras.optimizers.Adamax(learning_rate)
      case 'sgd': optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate,
                                        momentum=hp.Choice("momentum", [0.0, 0.95]),
                                        nesterov=hp.Boolean("nesterov"))

    
    activation = hp.Choice("activation", values=["swish", "gelu", 'elu'])

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(X_train.shape[1:]))
    model.add(tf.keras.layers.Flatten())

    for _ in range(n_hidden):
        model.add(tf.keras.layers.Dense(n_neurons, kernel_initializer='he_normal'))
        model.add(tf.keras.layers.BatchNormalization())
        model.add(tf.keras.layers.Activation(activation))
    
    model.add(tf.keras.layers.Dense(10, activation="softmax"))
    model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model

In [48]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=30,
    directory="fashion_mnist",
    project_name="params",
    seed=42,
)

tuner.search(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=5,
    batch_size=64,
)

Trial 30 Complete [00h 01m 13s]
val_accuracy: 0.8859999775886536

Best val_accuracy So Far: 0.8859999775886536
Total elapsed time: 00h 26m 53s


In [53]:
best_params = tuner.get_best_hyperparameters(1)[0]
best_params.values

{'n_hidden': 3,
 'n_neurons': 239,
 'learning_rate': 0.0008251039755551379,
 'optimizer': 'adamax',
 'momentum': 0.0,
 'nesterov': False,
 'activation': 'gelu'}

In [56]:
best_model = tuner.get_best_models(1)[0]

callbacks = [tf.keras.callbacks.EarlyStopping(patience=8)]

best_model.fit(X_train, y_train,
epochs=30, batch_size=64, validation_data=(X_valid, y_valid),
callbacks=callbacks)

/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamax', because it has 2 variables whereas the saved optimizer has 30 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Epoch 1/30
860/860 ━━━━━━━━━━━━━━━━━━━━ 17s 16ms/step - accuracy: 0.9224 - loss: 0.2105 - val_accuracy: 0.8854 - val_loss: 0.3191
Epoch 2/30
860/860 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9323 - loss: 0.1857 - val_accuracy: 0.8896 - val_loss: 0.3128
Epoch 3/30
860/860 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.9380 - loss: 0.1709 - val_accuracy: 0.8890 - val_loss: 0.3155
Epoch 4/30
860/860 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.9442 - loss: 0.1541 - val_accuracy: 0.8884 - val_loss: 0.3243
Epoch 5/30
860/860 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9509 - loss: 0.1370 - val_accuracy: 0.8892 - val_loss: 0.3394
Epoch 6/30
860/860 ━━━━━━━━━━━━━━━━━━━━ 15s 17ms/step - accuracy: 0.9537 - loss: 0.1277 - val_accuracy: 0.8884 - val_loss: 0.3330
Epoch 7/30
860/860 ━━━━━━━━━━━━━━━━━━━━ 13s 16ms/step - accuracy: 0.9564 - loss: 0.1185 - val_accuracy: 0.8610 - val_loss: 0.4388
Epoch 8/30
860/860 ━━━━━━━━━━━━━━━━━━━━ 14s 16ms/step - accuracy: 0.9628 - loss: 0.1065 - 

In [57]:
best_model.save('best_model_hw110.keras')

In [3]:
data = {}

In [6]:
best_model = tf.keras.models.load_model('best_model_hw110.keras')

start = time.time()
loss, acc = best_model.evaluate(X_test, y_test)
evaluate_time = time.time() - start

data['original'] = {'acc': acc, 'eval_time': evaluate_time}
acc, evaluate_time

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8800 - loss: 0.4150


(0.8799999952316284, 3.857299327850342)

In [7]:
# fold_BN_model = bn_convert()
# fold_BN_model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])


fold_BN_model = tf.keras.models.load_model('fold_BN_model.keras')

start = time.time()
loss, acc = fold_BN_model.evaluate(X_test, y_test)
evaluate_time = time.time() - start

data['fold_BN'] = {'acc': acc, 'eval_time': evaluate_time}
acc, evaluate_time

/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8800 - loss: 0.4150


(0.8799999952316284, 3.3144476413726807)

In [67]:
fold_BN_model.save('fold_BN_model.keras')

In [8]:
new_layers = []
for layer in fold_BN_model.layers:
    if isinstance(layer, tf.keras.layers.Activation):
        new_layers.append(tf.keras.layers.LeakyReLU(0.1))
    else:
        new_layers.append(layer)

lrelu_model = tf.keras.models.Sequential(new_layers)

In [9]:
lrelu_model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])

start = time.time()
loss, acc = lrelu_model.evaluate(X_test, y_test)
evaluate_time = time.time() - start

data['LeakyRelu'] = {'acc': acc, 'eval_time': evaluate_time}
acc, evaluate_time

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8781 - loss: 0.4639


(0.8780999779701233, 2.9697604179382324)

In [71]:
lrelu_model.save('lrelu_model.keras')

In [10]:
threshold = 0.01
counter = 0

for layer in lrelu_model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        w, b = layer.get_weights()

        mask = (np.abs(w) < threshold)
        w[mask] = 0
        counter += mask.sum()
        layer.set_weights([w, b])

counter

np.int64(19740)

In [12]:
sparce_model = deepcopy(lrelu_model)

/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
start = time.time()
loss, acc = sparce_model.evaluate(X_test, y_test)
evaluate_time = time.time() - start

data['sparse_model'] = {'acc': acc, 'eval_time': evaluate_time}
acc, evaluate_time

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8785 - loss: 0.4613


(0.8784999847412109, 2.868323802947998)

In [16]:
sparce_model.save('sparse_model.keras')

In [14]:
sparce_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 239)            │       187,615 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu (LeakyReLU)         │ (None, 239)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 239)            │        57,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_1 (LeakyReLU)       │ (None, 239)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 239)            │        57,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_2 (LeakyReLU)       │ (None, 239)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,400 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 609,472 (2.32 MB)

 Trainable params: 304,735 (1.16 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 304,737 (1.16 MB)

In [15]:
for layer in sparce_model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        w, b = layer.get_weights()
        print(w.shape, b.shape)
        print(np.sum(np.abs(w)))


(784, 239) (239,)
17983.812
(239, 239) (239,)
5399.3813
(239, 239) (239,)
5787.108
(239, 10) (10,)
295.00064


In [ ]:
sparse_model = tf.keras.models.load_model('sparse_model.keras')

new_layers = []
feature_inxs = None
n_neurons_keep = 220

input_shape = sparse_model.input_shape[1:]
new_layers.append(tf.keras.layers.InputLayer(input_shape=input_shape))

for layer in sparse_model.layers:
    if isinstance(layer, tf.keras.layers.Dense):
        w, b = layer.get_weights()

        if feature_inxs is not None:
            w = w[feature_inxs, :]

        w_sum = np.abs(w).sum(axis=0)
        inxs = np.sort(np.argsort(w_sum)[-n_neurons_keep:])
        feature_inxs = inxs

        new_w = w[:, inxs]
        new_b = b[inxs]

        new_dense = tf.keras.layers.Dense(len(inxs), input_dim=new_w.shape[0])
        new_dense.build((None, new_w.shape[0]))
        new_dense.set_weights([new_w, new_b])
        new_layers.append(new_dense)
        continue
    
    new_layers.append(layer)


reduced_neurons_model = tf.keras.Sequential(new_layers)

In [56]:
reduced_neurons_model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])

start = time.time()
loss, acc = reduced_neurons_model.evaluate(X_test, y_test)
evaluate_time = time.time() - start

data['reduced_neurons'] = {'acc': acc, 'eval_time': evaluate_time}
acc, evaluate_time

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8477 - loss: 1.1477


(0.8476999998092651, 1.9174044132232666)

In [59]:
reduced_neurons_model.save('reduced_neurons_model.keras')

In [60]:
reduced_neurons_model.layers

[<Flatten name=flatten, built=True>,
 <Dense name=dense_70, built=True>,
 <LeakyReLU name=leaky_re_lu, built=True>,
 <Dense name=dense_71, built=True>,
 <LeakyReLU name=leaky_re_lu_1, built=True>,
 <Dense name=dense_72, built=True>,
 <LeakyReLU name=leaky_re_lu_2, built=True>,
 <Dense name=dense_73, built=True>]

In [63]:
reduced_layers_model = tf.keras.models.Sequential(reduced_neurons_model.layers[:5] + reduced_neurons_model.layers[-1:])

In [64]:
reduced_layers_model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])

start = time.time()
loss, acc = reduced_layers_model.evaluate(X_test, y_test)
evaluate_time = time.time() - start

data['reduced_layers'] = {'acc': acc, 'eval_time': evaluate_time}
acc, evaluate_time

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.1124 - loss: 11.0570


(0.11240000277757645, 1.8125040531158447)

In [65]:
pd.DataFrame(data)

,original,fold_BN,LeakyRelu,sparse_model,reduced_neurons,reduced_layers
acc,0.880000,0.880000,0.87810,0.878500,0.847700,0.112400
eval_time,3.857299,3.314448,2.96976,2.868324,1.917404,1.812504
